# Cross-Entropy and the Softmax Gradient: One Loss for Every Classifier

The full code of [Cross-Entropy and the Softmax Gradient: One Loss for Every Classifier](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/softmax-and-cross-entropy) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/softmax-and-cross-entropy.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "softmax-and-cross-entropy"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `one_vs_rest.py`

The four-way softmax router against four independent sigmoids (one-vs-rest) on 20 random splits: validation accuracy, how far the sigmoid scores are from summing to 1, and validation cross-entropy.

In [ ]:
!{sys.executable} one_vs_rest.py

The post printed:

```text
val accuracy: softmax 0.621, one-vs-rest 0.624 (difference +0.003 ± 0.016)
one-vs-rest scores: sum is off from 1 by 0.19 on average; no route above 0.5 on 53% of questions (softmax: 33%)
val CE: softmax 0.891, one-vs-rest rescaled to sum to 1 0.885 (difference -0.006 ± 0.011)
```

## `info.py`

Entropy, cross-entropy and KL divergence for four models of one question's route distribution; ln 4 and ln 128,256; the logit gap and loss floor that label smoothing sets.

In [ ]:
!{sys.executable} info.py

The post printed:

```text
matches the truth  CE 0.940 = H 0.940 + KL 0.000   perplexity 2.56
uniform guess      CE 1.386 = H 0.940 + KL 0.446   perplexity 4.00
overconfident      CE 1.403 = H 0.940 + KL 0.462   perplexity 4.07
confidently wrong  CE 4.148 = H 0.940 + KL 3.207   perplexity 63.29

uniform over 4 routes: ln 4 = 1.386 nats; over a 128,256-token vocabulary: 11.76 nats
val CE 0.874 -> perplexity 2.40
smoothing 0.1: target (0.9250, 0.0250 x3), logit gap ln(0.9250/0.0250) = 3.61, loss floor 0.349
smoothing 0.01: target (0.9925, 0.0025 x3), logit gap ln(0.9925/0.0025) = 5.98, loss floor 0.052
```

## `softmax_basics.py`

`softmax` and `cross_entropy` from logits (the helper the other scripts import); run on its own, it checks the gradient p - y against finite differences and shows why the max is subtracted first.

In [ ]:
!{sys.executable} softmax_basics.py

The post printed:

```text
p - y:  [ 0.226   0.1746 -0.6219  0.2213]
largest gap to finite differences: 8.5e-11
naive softmax: [nan nan  0.  0.]   shifted: [0.7311 0.2689 0.     0.    ]
log(softmax(z)): [-0.31 -1.31 -inf -inf]   log-sum-exp: [-0.31 -1.31 -1000.31 -1005.31]
```

## `four_routes.py`

L2, no penalty, label smoothing and both, on 20 splits, paired against L2; then the model chosen on validation cross-entropy, scored on the pooled test splits with a confusion matrix, precision, recall and F1.

In [ ]:
!{sys.executable} four_routes.py

The post printed:

```text
                           val CE           vs L2  val acc           vs L2    ECE           vs L2
L2 0.003                    0.891                    0.621                  0.096                
no penalty                  0.976  +0.085 ± 0.054    0.633  +0.011 ± 0.021  0.154  +0.059 ± 0.033
smoothing 0.1               0.874  -0.017 ± 0.022    0.655  +0.034 ± 0.029  0.106  +0.010 ± 0.033
smoothing 0.1 + L2 0.003    0.907  +0.016 ± 0.014    0.628  +0.006 ± 0.013  0.125  +0.029 ± 0.031
smoothing + L2 vs smoothing alone: val CE +0.033 ± 0.018

chosen on validation CE: smoothing 0.1. test CE 0.848 (perplexity 2.34)
test confusion, pooled over the 20 splits (rows = true route):
                    serving architecture       agents  foundations
serving                 120           29           22           29
architecture             19          113           12           56
agents                   10           10          153           27
foundations              17           19           20          144
serving        precision 0.723  recall 0.600  F1 0.656
architecture   precision 0.661  recall 0.565  F1 0.609
agents         precision 0.739  recall 0.765  F1 0.752
foundations    precision 0.562  recall 0.720  F1 0.632
test accuracy 0.6625; mean recall 0.6625; macro-F1 0.6621
```

## `longer_training.py`

The same four models trained for 4,000 and 16,000 steps: validation and training cross-entropy, weight norm, mean top probability.

In [ ]:
!{sys.executable} longer_training.py

The post printed:

```text
                           steps  val CE           vs L2    |W|  train CE  mean top p  val acc
L2 0.003                   4,000   0.891                   10.6     0.131       0.622    0.621
L2 0.003                  16,000   0.891                   10.6     0.131       0.622    0.621
no penalty                 4,000   0.976  +0.085 ± 0.054   21.7     0.008       0.776    0.633
no penalty                16,000   1.068  +0.177 ± 0.078   27.3     0.002       0.815    0.630
smoothing 0.1              4,000   0.874  -0.017 ± 0.022   14.1     0.080       0.616    0.655
smoothing 0.1             16,000   0.876  -0.015 ± 0.025   14.7     0.078       0.613    0.661
smoothing 0.1 + L2 0.003   4,000   0.907  +0.016 ± 0.014    9.0     0.200       0.563    0.628
smoothing 0.1 + L2 0.003  16,000   0.907  +0.016 ± 0.014    9.0     0.200       0.563    0.628
```

## `read_weights.py`

One split's learned weights per route, its bias, and the two architecture questions it sends to foundations, word by word.

In [ ]:
!{sys.executable} read_weights.py

The post printed:

```text
serving       vllm +1.11, batching +1.11, token +0.98, paged +0.96, to +0.95, quantization +0.92
architecture  attention +1.92, unembedding +1.61, residual +1.59, stream +1.29, transformers +1.21, layer +1.17
agents        agent +1.83, prompt +1.63, context +1.47, calling +1.38, rot +1.33, retrieval +1.22
foundations   backpropagation +1.74, confusion +1.29, learning +1.23, gradients +1.23, sigmoid +1.12, optimizer +1.09

bias b = [ 0.13 -0.17 -0.17  0.21], softmax(b) = [0.28 0.21 0.21 0.3 ]

architecture -> foundations: How does rotary position embedding encode distance?
  never seen in training: rotary, position, embedding, encode, distance
  how   weights [-0.7   0.19  0.27  0.24]
  does  weights [ 0.43 -0.2  -0.01 -0.23]
  words alone: p = [0.19 0.24 0.32 0.25]   with the bias: p = [0.22 0.21 0.27 0.31]

architecture -> foundations: How is the embedding table trained?
  never seen in training: embedding, table, trained
  how   weights [-0.7   0.19  0.27  0.24]
  is    weights [ 0.04 -0.24 -0.15  0.34]
  the   weights [ 0.02  0.57 -0.43 -0.16]
  words alone: p = [0.12 0.38 0.16 0.34]   with the bias: p = [0.13 0.31 0.14 0.41]
```

## `unsure.py`

Across 20 splits: how often the router's top probability is below 0.4, how often those questions are right, where they go, and whether an even bias would move them.

In [ ]:
!{sys.executable} unsure.py

The post printed:

```text
test questions with top probability < 0.4: 16%, right 40% of the time (the rest: 71%)
where they go:   serving 18%, architecture 23%, agents 22%, foundations 37%
all predictions: serving 21%, architecture 21%, agents 26%, foundations 32%
share of words never seen in training: unsure 33%, the rest 26%
of the 47 unsure questions sent to foundations, 24 go elsewhere with an even bias
questions containing 'what': serving 11, architecture 19, agents 16, foundations 28
questions containing 'is': serving 13, architecture 17, agents 14, foundations 28
```

## `logit_memory.py`

Memory for the logits of one training batch at Llama 3 8B's 128,256-token vocabulary, against the hidden states and checkpointed activations.

In [ ]:
!{sys.executable} logit_memory.py

The post printed:

```text
B=1, T=8192: 8,192 positions x 128,256 = 1.05 B logits; FP32 3.9 GiB, and the same again for their gradient
B=8, T=8192: 65,536 positions x 128,256 = 8.41 B logits; FP32 31.3 GiB, and the same again for their gradient
the hidden states that produce them, BF16: 0.50 GiB
with activation checkpointing, the saved input of each of 32 layers: 16 GiB
```